# BrainTart — Attention U-Net 3D for BraTS 2026 Inpainting

This notebook clones the BrainTart repository, downloads the challenge data,
trains the Attention U-Net 3D model, runs inference, and evaluates locally.

**Hardware target:** 2x T4 (16 GB each) on Kaggle, PyTorch DDP.

### Sections
1. Environment Setup & Repo Clone
2. Synapse Auth & Data Download
3. Configuration Overview
4. Training
5. Inference & Submission Generation
6. Local Evaluation
7. Submission Checklist

## 1. Environment Setup & Repo Clone

In [ ]:
# Install dependencies and clone the BrainTart repo
import subprocess, sys, os

# Clone the repo (replace with your actual repo URL)
REPO_URL = "https://github.com/YOUR_USERNAME/BrainTart.git"  # TODO: set your repo URL
REPO_DIR = "/kaggle/working/BrainTart"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    print(f"Cloned BrainTart to {REPO_DIR}")
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
    print(f"Updated BrainTart at {REPO_DIR}")

# Install Python dependencies
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r",
     os.path.join(REPO_DIR, "requirements.txt")]
)
print("Dependencies installed.")

In [ ]:
# Add repo to sys.path so imports work
import sys
sys.path.insert(0, REPO_DIR)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPUs     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {props.name}  {props.total_memory/1e9:.1f} GB")

## 2. Synapse Auth & Data Download

In [ ]:
import zipfile
from pathlib import Path
import synapseclient
from kaggle_secrets import UserSecretsClient

# Synapse auth via Kaggle Secrets (secret name: "brats")
SYNAPSE_TOKEN = UserSecretsClient().get_secret("brats")
syn = synapseclient.Synapse()
syn.login(authToken=SYNAPSE_TOKEN)
print("Logged in to Synapse.")

# Download the BraTS 2026 inpainting training data
DOWNLOAD_DIR = Path("/kaggle/working/brats_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

entity = syn.get(entity="syn51523038", version=1, downloadLocation=str(DOWNLOAD_DIR))
filepath = entity.path
print(f"Downloaded to: {filepath}")

In [ ]:
# Extract dataset
DATASET_ROOT = Path("/kaggle/working/brats_data")
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

if not any(DATASET_ROOT.rglob("BraTS-GLI-*-*")):
    print("Extracting dataset...")
    with zipfile.ZipFile(filepath, "r") as zf:
        zf.extractall(DATASET_ROOT)
    candidates = list(DATASET_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        DATASET_ROOT = candidates[0].parent
else:
    candidates = list(DATASET_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        DATASET_ROOT = candidates[0].parent
    print("Dataset already extracted.")

cases = [p for p in DATASET_ROOT.iterdir() if p.is_dir()]
print(f"DATASET_ROOT : {DATASET_ROOT}")
print(f"Cases found  : {len(cases)}")

## 3. Configuration Overview

In [ ]:
from configs import Config

cfg = Config()
cfg.DATASET_ROOT = DATASET_ROOT
cfg.makedirs()

# Override any defaults here:
# cfg.NUM_EPOCHS = 100
# cfg.BATCH_PER_GPU = 1

print(f"Crop shape     : {cfg.CROP_SHAPE}")
print(f"Base channels  : {cfg.BASE_CHANNELS}")
print(f"Depth          : {cfg.DEPTH}")
print(f"Effective batch: {cfg.BATCH_PER_GPU * max(torch.cuda.device_count(), 1)}")
print(f"Epochs         : {cfg.NUM_EPOCHS}")

## 4. Training

Runs DDP on available GPUs via the `train.py` script.

In [ ]:
import subprocess, sys

train_cmd = [
    sys.executable, f"{REPO_DIR}/train.py",
    "--dataset", str(DATASET_ROOT),
    "--epochs", str(cfg.NUM_EPOCHS),
    "--batch", str(cfg.BATCH_PER_GPU),
    "--lr", str(cfg.LR),
    "--seed", str(cfg.SEED),
    "--crop", str(cfg.CROP_SHAPE[0]), str(cfg.CROP_SHAPE[1]), str(cfg.CROP_SHAPE[2]),
    "--base-ch", str(cfg.BASE_CHANNELS),
    "--depth", str(cfg.DEPTH),
    "--checkpoint-dir", str(cfg.CHECKPOINT_DIR),
    "--results-dir", str(cfg.RESULTS_DIR),
    "--output-dir", str(cfg.OUTPUT_DIR),
]

print("Running:", " ".join(train_cmd))
result = subprocess.run(train_cmd, capture_output=False)
print(f"Training exited with code {result.returncode}")

In [ ]:
# Display the training curve
from IPython.display import Image, display
from pathlib import Path

curve_path = cfg.OUTPUT_DIR / "training_curve_final.png"
if curve_path.exists():
    display(Image(filename=str(curve_path)))
else:
    print("Training curve not found — check training output above.")

## 5. Inference & Submission Generation

Output format: `BraTS-GLI-XXXXX-YYY-t1n-inference.nii.gz` (240x240x155)

In [ ]:
import subprocess, sys

checkpoint = cfg.CHECKPOINT_DIR / "best_model.pt"
if not checkpoint.exists():
    # Fallback to latest periodic checkpoint
    ckpts = sorted(cfg.CHECKPOINT_DIR.glob("ckpt_epoch_*.pt"))
    checkpoint = ckpts[-1] if ckpts else None

if checkpoint:
    infer_cmd = [
        sys.executable, f"{REPO_DIR}/inference.py",
        "--dataset", str(DATASET_ROOT),
        "--checkpoint", str(checkpoint),
        "--results-dir", str(cfg.RESULTS_DIR),
        "--crop", str(cfg.CROP_SHAPE[0]), str(cfg.CROP_SHAPE[1]), str(cfg.CROP_SHAPE[2]),
        "--base-ch", str(cfg.BASE_CHANNELS),
        "--depth", str(cfg.DEPTH),
    ]
    print("Running:", " ".join(infer_cmd))
    subprocess.run(infer_cmd)
else:
    print("ERROR: No checkpoint found. Train first.")

## 6. Local Evaluation

Mirrors the Synapse server evaluation: SSIM, PSNR, MSE on mask-healthy region,
normalised by max(t1n_voided).

In [ ]:
import subprocess, sys

eval_cmd = [
    sys.executable, f"{REPO_DIR}/evaluate.py",
    "--dataset", str(DATASET_ROOT),
    "--results", str(cfg.RESULTS_DIR),
    "--max-cases", "30",  # Quick sanity check; remove for full eval
]
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd)

## 7. Submission Checklist & Upload

Run this before uploading to Synapse.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, f"{REPO_DIR}/evaluate.py",
    "--dataset", str(DATASET_ROOT),
    "--results", str(cfg.RESULTS_DIR),
    "--checklist",
])

print("\nTo upload to Synapse:")
print(f"  cd {cfg.RESULTS_DIR}")
print(f"  zip -j results.zip *.nii.gz")
print("  Then upload results.zip on the Synapse challenge page.")